# Bank Marketing (TabFairGAN) 

Author: Ilse Harmers \
Last updated: February 24, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
from tabfairgan import TFG
import utils
import torch
device = torch.device("cuda:0" if (torch.cuda.is_available() and 1 > 0) else "cpu")
print(device)

In [ ]:
# Importing train set.
bank_train = pd.read_csv("./train-test-datasets/bank-marketing/bank_train.csv")

# Preprocessing Bank for TabFairGAN by binarizing the 'age' column. Age less than or equal to 25 is privileged (= 1); age more than 25 
# unprivileged (= 0). 
bank_train.loc[bank_train["age"] <= 25, "age"] = 1
bank_train.loc[bank_train["age"] > 25, "age"] = 0
bank_train["age"] = bank_train["age"].astype(str)   # TabFairGAN only accepts string-based column for the sensitive attribute

print(bank_train.columns.to_list())
bank_train.describe()

In [ ]:
# Setting up GAN training parameters. 
fairness_config = {
    'fair_epochs': 25,
    'lamda': 0.5,
    'S': 'age',
    'Y': 'y',
    'S_under': "0",
    'Y_desire': "yes"
}

In [ ]:
r = 1
while r < 6:
    
    print(f"Run {r}")
    
    # Synthesizing the dataset with TabFairGAN. 
    tfg = TFG(bank_train, epochs = 100, batch_size = 256, device = "cuda:0", fairness_config = fairness_config)
    # Training TabFairGAN synthesizer. 
    tfg.train()

    try:
        # Generating the first synthetic dataset.
        sample1 = tfg.generate_fake_df(bank_train.shape[0])
        sample1["age"] = sample1["age"].astype(int)   # reversing age column back to int 
        # Encoding the target attribute for the fairness analysis.
        y1_encoded = utils.one_hot_encode(sample1[["y"]], order = [["no", "yes"]])
        age1_encoded = sample1["age"].copy()
        sample1_encoded = pd.concat([age1_encoded.reset_index(drop = True), y1_encoded.reset_index(drop = True)], axis = 1)
        dem1 = utils.demographic_parity(df = sample1_encoded, s = "age", y = "y")
        dis1 = utils.disparate_impact(df = sample1_encoded, s = "age", y = "y")
        print("Sample 1 [dem., dis.]: ",  [dem1, dis1])
        
        # Generating the second synthetic dataset.
        sample2 = tfg.generate_fake_df(bank_train.shape[0])
        sample2["age"] = sample2["age"].astype(int)   # reversing age column back to int
        # Encoding the target attribute for the fairness analysis.
        y2_encoded = utils.one_hot_encode(sample2[["y"]], order = [["no", "yes"]])
        age2_encoded = sample2["age"].copy()
        sample2_encoded = pd.concat([age2_encoded.reset_index(drop = True), y2_encoded.reset_index(drop = True)], axis = 1)
        dem2 = utils.demographic_parity(df = sample2_encoded, s = "age", y = "y")
        dis2 = utils.disparate_impact(df = sample2_encoded, s = "age", y = "y")
        print("Sample 2 [dem., dis.]: ",  [dem2, dis2])

        # Generating the third synthetic dataset.
        sample3 = tfg.generate_fake_df(bank_train.shape[0])
        sample3["age"] = sample3["age"].astype(int)   # reversing age column back to int 
        # Encoding the target attribute for the fairness analysis.
        y3_encoded = utils.one_hot_encode(sample3[["y"]], order = [["no", "yes"]])
        age3_encoded = sample3["age"].copy()
        sample3_encoded = pd.concat([age3_encoded.reset_index(drop = True), y3_encoded.reset_index(drop = True)], axis = 1)
        dem3 = utils.demographic_parity(df = sample3_encoded, s = "age", y = "y")
        dis3 = utils.disparate_impact(df = sample3_encoded, s = "age", y = "y")
        print("Sample 3 [dem., dis.]: ",  [dem3, dis3])

        # Saving the synthetic datasets.
        sample1.to_csv(f"./synthetic-datasets_TabFair/bank-marketing/run{r}/sample1.csv", index = False)
        sample2.to_csv(f"./synthetic-datasets_TabFair/bank-marketing/run{r}/sample2.csv", index = False)
        sample3.to_csv(f"./synthetic-datasets_TabFair/bank-marketing/run{r}/sample3.csv", index = False)
        
        r += 1
    except ZeroDivisionError:
        r += 0